In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import dtale


bureau_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau.train-cleaned.parquet")

load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)

In [ ]:
bureau_df["credit_active"].value_counts()


credit_active
Closed    919144
Active    540410
Sold        4168
Name: count, dtype: int64

In [2]:
bureau_df.sort_values(["id_curr", "days_credit"],inplace=True,ascending=False)
last_three = bureau_df.groupby("id_curr").head(3)
last_three = last_three.copy()
last_three["loan_order"] = last_three.groupby("id_curr").cumcount() + 1
last_three_columns = last_three.pivot(index="id_curr", columns="loan_order")
last_three_columns.columns =[f"{col}_prev_{rank}" for col, rank in last_three_columns.columns]


dtale.show(last_three)


2026-06-04 01:51:01,286 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "/Users/dreamcast/Documents/Home-Credit-Default-Risk-Kaggle/env/lib/python3.11/site-packages/flask/app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dreamcast/Documents/Home-Credit-Default-Risk-Kaggle/env/lib/python3.11/site-packages/flask/app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dreamcast/Documents/Home-Credit-Default-Risk-Kaggle/env/lib/python3.11/site-packages/flask/app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dreamcast/Documents/Home-Credit-Default-Risk-Kaggle/env/lib/python3.11/site-packages/flask/app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [10]:
bureau_df["credit_active"]=bureau_df["credit_active"].str.lower()
bureau= pd.get_dummies(bureau_df, columns=["credit_active"])
bureau.head()

,id_curr,id_bureau,credit_currency,days_credit,flag_have_credit_day_overdue,days_credit_enddate,DAYS_CREDIT_ENDDATE,days_credit_enddate_first_cluster_values,days_credit_enddate_is_present,days_credit_enddate_closed,...,amt_credit_sum_limit_long_limit,have_amt_credit_sum_overdue,have_amt_credit_sum_overdue_is_present,credit_type,days_credit_update,amt_annuity,flag_is_present_amt_annuity,credit_active_active,credit_active_closed,credit_active_sold
0,215354,5714462,currency 1,-497,0,-153.0,NaN,-153.0,1,1,...,0,0,0,Consumer credit,-131,NaN,1,False,True,False
1,215354,5714463,currency 1,-208,0,1075.0,NaN,1075.0,1,0,...,0,0,0,Credit card,-20,NaN,1,True,False,False
2,215354,5714464,currency 1,-203,0,528.0,NaN,528.0,1,0,...,0,0,0,Consumer credit,-16,NaN,1,True,False,False
3,215354,5714465,currency 1,-203,0,NaN,NaN,NaN,0,0,...,0,0,0,Credit card,-16,NaN,1,True,False,False
4,215354,5714466,currency 1,-629,0,1197.0,NaN,1197.0,1,0,...,0,0,0,Consumer credit,-21,NaN,1,True,False,False


In [3]:
# In the case that there are multiple modes, we use the most resent 
def get_first_mode(x):
    mode_series = x.mode()
    return mode_series.iloc[0] if not mode_series.empty else pd.NA

bureau_aggregattted = bureau_df.groupby("id_curr").agg({
    "id_curr": ["count"],
    "id_bureau": ["count"],
    "flag_have_credit_day_overdue": ["count"],
    "credit_type": [get_first_mode],
    "amt_credit_sum_limit": ["max"],
    "cnt_credit_prolong": ["max", "min", "mean"],
    "amt_credit_sum": ["max", "min", "mean"],
    
})



dtale.show(bureau_aggregattted)


In [ ]:
combined_rows = pd.concat(bureau_aggregattted, last_three_columns,  axis=1 )
dtale.show(combined_rows)


MergeError: Not allowed to merge between different levels. (2 levels on the left, 1 on the right)